# Advanced Wi-Fi Track Data Clean

This method re-clean data according to track repeat frequency ( caused by unstable signal appears among multiple trackers) based on given basic cleaned data.

Generate **virtual wifi tracker** after cleaning.(Virtual wifi trackers' id will be under zero)

In [37]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mat
import matplotlib.cm as cm
import os
import datetime
import time
from tqdm import tqdm
import plotly.express as px
import plotly.io as pio

# 设置plotly默认主题
pio.templates.default = 'plotly_white'

mat.rcParams['font.family'] = 'SimHei'
mat.rcParams['font.sans-serif'] = 'SimHei'

import warnings
warnings.filterwarnings('ignore')

## Read Data

In [2]:
df_read = pd.read_csv(os.getcwd()+'dacang/cleaned_data/dacang_cleaned_data_basic.csv')
df = df_read.copy()
df_read

,a,r,t,m
0,22,-43,2022-07-05 23:49:36,"48,74,38,155,214,98"
1,22,-35,2022-07-05 23:49:36,"252,107,240,89,142,53"
2,22,-42,2022-07-05 23:49:36,"164,125,159,153,152,106"
3,22,-38,2022-07-05 23:49:47,"252,107,240,89,142,53"
4,22,-42,2022-07-05 23:49:47,"164,125,159,153,152,106"
...,...,...,...,...
1916096,66,-49,2022-08-07 23:44:30,"88,102,186,101,140,144"
1916097,66,-49,2022-08-07 23:46:54,"88,102,186,101,140,144"
1916098,66,-48,2022-08-07 23:48:38,"88,102,186,101,140,144"
1916099,66,-49,2022-08-07 23:51:14,"88,102,186,101,140,144"


In [3]:
#label each mac with date
pbar = tqdm(total=len(df))
df['oriMac'] = df['m']
df.t = pd.to_datetime(df.t)
df['calender'] = df.t.apply(lambda x : str(x.month)+'.'+str(x.day))
for index,row in df.iterrows():
    df.loc[index,'m'] = str(row['calender'])+'-'+row.m
    pbar.update(1)
pbar.close()
df

  0%|          | 0/1916101 [00:00<?, ?it/s]

100%|██████████| 1916101/1916101 [02:24<00:00, 13217.61it/s]


,a,r,t,m,oriMac,calender
0,22,-43,2022-07-05 23:49:36,"7.5-48,74,38,155,214,98","48,74,38,155,214,98",7.5
1,22,-35,2022-07-05 23:49:36,"7.5-252,107,240,89,142,53","252,107,240,89,142,53",7.5
2,22,-42,2022-07-05 23:49:36,"7.5-164,125,159,153,152,106","164,125,159,153,152,106",7.5
3,22,-38,2022-07-05 23:49:47,"7.5-252,107,240,89,142,53","252,107,240,89,142,53",7.5
4,22,-42,2022-07-05 23:49:47,"7.5-164,125,159,153,152,106","164,125,159,153,152,106",7.5
...,...,...,...,...,...,...
1916096,66,-49,2022-08-07 23:44:30,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7
1916097,66,-49,2022-08-07 23:46:54,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7
1916098,66,-48,2022-08-07 23:48:38,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7
1916099,66,-49,2022-08-07 23:51:14,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7


In [4]:
len(df.m.unique())

31798

In [5]:
#删除打上标签后只出现过一次的mac
df = df[df.duplicated('m',keep=False)]
df

,a,r,t,m,oriMac,calender
0,22,-43,2022-07-05 23:49:36,"7.5-48,74,38,155,214,98","48,74,38,155,214,98",7.5
1,22,-35,2022-07-05 23:49:36,"7.5-252,107,240,89,142,53","252,107,240,89,142,53",7.5
2,22,-42,2022-07-05 23:49:36,"7.5-164,125,159,153,152,106","164,125,159,153,152,106",7.5
3,22,-38,2022-07-05 23:49:47,"7.5-252,107,240,89,142,53","252,107,240,89,142,53",7.5
4,22,-42,2022-07-05 23:49:47,"7.5-164,125,159,153,152,106","164,125,159,153,152,106",7.5
...,...,...,...,...,...,...
1916096,66,-49,2022-08-07 23:44:30,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7
1916097,66,-49,2022-08-07 23:46:54,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7
1916098,66,-48,2022-08-07 23:48:38,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7
1916099,66,-49,2022-08-07 23:51:14,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7


## 保存或读取已打上date标签的dataframe

In [2]:
#保存
#df.to_csv(os.getcwd()+"/dacang/cleaned_data/dacang_cleaned_data_dataLabeled.csv",index=False)
#读取
df = pd.read_csv(os.getcwd()+"/dacang/cleaned_data/dacang_cleaned_data_dataLabeled.csv")
df

,a,r,t,m,oriMac,calender
0,22,-43,2022-07-05 23:49:36,"7.5-48,74,38,155,214,98","48,74,38,155,214,98",7.5
1,22,-35,2022-07-05 23:49:36,"7.5-252,107,240,89,142,53","252,107,240,89,142,53",7.5
2,22,-42,2022-07-05 23:49:36,"7.5-164,125,159,153,152,106","164,125,159,153,152,106",7.5
3,22,-38,2022-07-05 23:49:47,"7.5-252,107,240,89,142,53","252,107,240,89,142,53",7.5
4,22,-42,2022-07-05 23:49:47,"7.5-164,125,159,153,152,106","164,125,159,153,152,106",7.5
...,...,...,...,...,...,...
1899913,66,-49,2022-08-07 23:44:30,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7
1899914,66,-49,2022-08-07 23:46:54,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7
1899915,66,-48,2022-08-07 23:48:38,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7
1899916,66,-49,2022-08-07 23:51:14,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7


## 以天为单位清理异常mac

In [3]:
#获取每个mac一天中经过的探针数：a_count
count_list = df.groupby('m').a.value_counts()
df_count = count_list.to_frame().reset_index()
a_count = df_count.m.value_counts()
#获取每个mac一天中的数据量:s_count
count_list = df.m.value_counts()


In [4]:
#获取每个mac一天中的持续时间
df.t = pd.to_datetime(df.t)
df

,a,r,t,m,oriMac,calender
0,22,-43,2022-07-05 23:49:36,"7.5-48,74,38,155,214,98","48,74,38,155,214,98",7.5
1,22,-35,2022-07-05 23:49:36,"7.5-252,107,240,89,142,53","252,107,240,89,142,53",7.5
2,22,-42,2022-07-05 23:49:36,"7.5-164,125,159,153,152,106","164,125,159,153,152,106",7.5
3,22,-38,2022-07-05 23:49:47,"7.5-252,107,240,89,142,53","252,107,240,89,142,53",7.5
4,22,-42,2022-07-05 23:49:47,"7.5-164,125,159,153,152,106","164,125,159,153,152,106",7.5
...,...,...,...,...,...,...
1899913,66,-49,2022-08-07 23:44:30,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7
1899914,66,-49,2022-08-07 23:46:54,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7
1899915,66,-48,2022-08-07 23:48:38,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7
1899916,66,-49,2022-08-07 23:51:14,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7


In [5]:
mac_list = df.m.unique()
len(mac_list)

15615

In [7]:
#如
def GetDuration(df_now):
    time = df_now.iloc[len(df_now)-1].t - df_now.iloc[0].t
    time = time.components.hours+(time.components.minutes/60)
    return round(time,2)

macDur_dict = {}

for mac in tqdm(mac_list):
    df_now = df[df.m == mac]
    time = GetDuration(df_now)
    macDur_dict.update({mac:time})

macDur_dict

  0%|          | 65/15615 [00:03<15:51, 16.35it/s]


KeyboardInterrupt: 

In [6]:
#保存或读取macDur_dict
#保存
#dict_str = str(macDur_dict)
#with open('../share/dict_file.txt', 'w') as file:
#    file.write(dict_str)

#读取
with open('../share/dict_file.txt', 'r') as file:
    dict_str = file.read()
    macDur_dict = eval(dict_str)
mac_list = list(macDur_dict.keys())
mac_dur_list = list(macDur_dict.values())

In [7]:
#生成统计数值的dataframe
df_sta = pd.DataFrame({'M':[],'Dur':[],'A_count':[],'Count':[],'oriMac':[],'date':[]})
for i in range(len(mac_list)):
    df_sta = df_sta._append({'M':mac_list[i],
                             'Dur':mac_dur_list[i],
                             'A_count':a_count[mac_list[i]],
                             'Count':count_list[mac_list[i]],
                             'oriMac':mac_list[i].split('-')[1],
                             'date':mac_list[i].split('-')[0]},ignore_index=True)
df_sta

,M,Dur,A_count,Count,oriMac,date
0,"7.5-48,74,38,155,214,98",21.83,6.0,147.0,"48,74,38,155,214,98",7.5
1,"7.5-252,107,240,89,142,53",17.02,2.0,37.0,"252,107,240,89,142,53",7.5
2,"7.5-164,125,159,153,152,106",17.02,2.0,19.0,"164,125,159,153,152,106",7.5
3,"7.5-112,138,9,131,175,199",17.02,2.0,47.0,"112,138,9,131,175,199",7.5
4,"7.5-208,151,254,40,222,169",17.02,3.0,11.0,"208,151,254,40,222,169",7.5
...,...,...,...,...,...,...
15610,"8.7-172,92,102,108,72,204",0.00,1.0,2.0,"172,92,102,108,72,204",8.7
15611,"8.7-172,186,190,231,156,25",0.00,1.0,2.0,"172,186,190,231,156,25",8.7
15612,"8.7-216,138,220,202,211,84",1.93,1.0,8.0,"216,138,220,202,211,84",8.7
15613,"8.7-244,76,112,14,81,63",0.03,1.0,3.0,"244,76,112,14,81,63",8.7


In [8]:
fig = px.scatter(df_sta,x="Dur",y="A_count",size="Count",color="oriMac",
                 hover_name="M")
fig.update_xaxes(title='24h内持续时间')
fig.update_yaxes(title="到达的探针数量")
fig.show()

**偶发信号**和和**多探针覆盖范围下的智能设备信号**是需要清洗的内容，
- 偶发信号的特征为数据量极少且到达探针的数量少
- 多探针覆盖范围下的智能设备信号特征为数据量多且到达探针的数量为1-4个


另外主要还有**常驻人群**，**游客**，**通勤人群**三类数据，但由于信号干扰等原因，不同数据之间的分隔基准并不明确，同时人群种类可能并不仅仅只有前文所列的三类，不同类型也有可能因为信号干扰、行为模式不同而导致数据差异，因此通过简单的逻辑判断清洗效果和分类效果都不甚理想。

尝试通过kmeans或dbscan聚类来进行分类

## DBSCAN聚类
DBSCAN（Density-Based Spatial Clustering of Applications with Noise）是一种基于密度的聚类算法，可以将数据点分成不同的簇，并且能够识别噪声点（不属于任何簇的点）。

DBSCAN聚类算法的基本思想是：

在给定的数据集中，根据每个数据点周围其他数据点的密度情况，将数据点分为核心点、边界点和噪声点。

- 核心点是周围某个半径内有足够多其他数据点的数据点；
- 边界点是不满足核心点要求，但在某个核心点的半径内的数据点；
- 噪声点则是不满足任何条件的点。


In [9]:
df_cluster = df_sta.reindex(columns = ['Dur','A_count','Count'])
df_cluster

,Dur,A_count,Count
0,21.83,6.0,147.0
1,17.02,2.0,37.0
2,17.02,2.0,19.0
3,17.02,2.0,47.0
4,17.02,3.0,11.0
...,...,...,...
15610,0.00,1.0,2.0
15611,0.00,1.0,2.0
15612,1.93,1.0,8.0
15613,0.03,1.0,3.0


In [10]:
def EliminateNoise(df,labels):
    df_result = df.copy()
    df_result['label'] = labels
    df_result = df_result[df_result.label != -1]
    return df_result

### 轮廓系数与卡林实际-哈拉巴斯指数进行评估

- **轮廓系数（silhouette_score）**，轮廓系数指标不关注样本的实际类别，而是通过分析聚类结果中样本的内聚度和分离度两种因素来给出成绩，取值范围为（-1，1），**值越大代表聚类的结果越合理**。
- **卡林斯基-哈拉巴斯指数(Calinski-Harabasz Index)**，组内离散程度低，**协方差矩阵**的迹就会越小，同时，组间离散程度大，**协方差矩阵**的迹就会越大。因此calinski_harabasz**指数越高越好**。
  
 $$ s(k)=\frac{T_r(B_k)}{T_r(W_k)}*\frac{N-k}{k-1} $$

其中$N$是数据集的样本量，$k$为簇的个数，$B_k$是组间离散矩阵,即不同簇之间的协方差矩阵。$W_k$是簇内离散矩阵，即一个簇内数据的协方差矩阵，而$Tr$表示矩阵的迹。


In [11]:
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score # 轮廓系数评估函数
from sklearn.metrics import calinski_harabasz_score #卡林斯基-哈拉巴斯指数


In [46]:
#确定半径 eps
init_eps = 1
eps = []
min_samples = []
cluster_count = []

cluster_labels = []

silhouette = []
calinski_harabasz = []

noise_num = []

for i in tqdm(range(100)):
    for j in range(15):
    #eps表示样本点的领域半径，min_samples表示样本点在领域半径内的最小数量(一般取2*dim = 6)
        dbscan = DBSCAN(eps=init_eps + i, min_samples=1+j)
        cluster = dbscan.fit(df_cluster)
        df_result = EliminateNoise(df_cluster,cluster.labels_)
        eps.append(init_eps + i)
        min_samples.append(1+j)
        
        if len(set(cluster.labels_)) <= 2:
            cluster_count.append(0)
            cluster_labels.append([0])
            silhouette.append(0)
            calinski_harabasz.append(0)
            noise_num.append(0)
        else:
            cluster_count.append(len(set(cluster.labels_)) - (1 if -1 in cluster.labels_ else 0))#减去噪声聚类
            labels = []
            for i in range(len(cluster.labels_)):
                labels.append(cluster.labels_[i])
            
            cluster_labels.append(labels)
            silhouette.append(silhouette_score(df_result, df_result.label))
            calinski_harabasz.append( calinski_harabasz_score(df_result, df_result.label))
            noise_num.append(np.count_nonzero(cluster.labels_ == -1))

100%|██████████| 100/100 [1:12:17<00:00, 43.38s/it]


In [63]:
for i in range(len(cluster_labels)):
    if cluster_labels[i] == [0]:
        cluster_labels[i] = [0]*15615

In [64]:
#保存聚类数据
df_dbscan_result = pd.DataFrame({"cluster_count":cluster_count,"silhouette":silhouette,"calinski_harabasz":calinski_harabasz,"noise_num":noise_num})
df_dbscan_result.to_csv(os.getcwd()+"/dacang/cluster/dbscan_result_eliNoise.csv",index=False)
np.save(os.getcwd()+"/dacang/cluster/cluster_mat_result.npy",cluster_labels)
np.save(os.getcwd()+"/dacang/cluster/cluster_mat_eps.npy",eps)
np.save(os.getcwd()+"/dacang/cluster/cluster_mat_min_samples.npy",min_samples)

#读取聚类数据
#df_dbscan_result = pd.read_csv(os.getcwd()+"/dacang/cluster/dbscan_result.csv")
#cluster_count = df_dbscan_result.cluster_count
#silhouette = df_dbscan_result.silhouette
#calinski_harabasz = df_dbscan_result.calinski_harabasz
#noise_num = df_dbscan_result.noise_num
#eps = []
#for i in (range(100)):
#    eps.append(1 + i)

In [46]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

colors = ['rgb(67,67,67)', 'rgb(115,115,115)', 'rgb(189,189,189)']
line_size = [2, 2, 2]
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(
    go.Scatter(x = eps, y = silhouette,
                         name='轮廓系数',
                         mode='lines',
                         line=dict(color=colors[1],width=line_size[1]),
                         ),
                         secondary_y = False)
fig.add_trace(
    go.Scatter(x = eps, y = calinski_harabasz,
                         name='卡林斯基-哈拉巴斯指数',
                         mode='lines',
                         line=dict(color=colors[0],width=line_size[2]),
                         ),
                         secondary_y = True)
fig.update_yaxes(title_text="<b>轮廓系数</b> ", secondary_y=False)
fig.update_yaxes(title_text="<b>卡林斯基-哈拉巴斯指数</b>", secondary_y=True)
fig.update_layout(
    width = 800,
    xaxis=dict(
        title_text = "<b>领域半径(eps)</b>",
        showline=True,
        showgrid=False,
        showticklabels=True,
        linecolor='rgb(204, 204, 204)',
        linewidth=2,
        ticks='outside',
        tickfont=dict(
            family='Arial',
            size=12,
            color='rgb(82, 82, 82)',
        ),
    ),
)
fig.show()



轮廓系数与卡林斯基-哈拉巴斯指数都是内部聚类评价指标，即在不知道真实labels时对聚类结果进行评价。

内部聚类评价指标的一个通病是默认数据集中的类别是簇状结构。

事实上，真实世界的数据集很多都不是簇状结构，所以内部聚类评价往往并不客观。

对于非簇状结构的聚类结果的评价往往使用外部聚类评价指标，常见的有NMI，AMI，F-score，Accuracy等。

In [47]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(x = eps, y = cluster_count,
                         name='聚类数量',
                         mode='lines',
                         line=dict(color=colors[0],width=line_size[0]),
                         ),
                         )
fig.update_layout(
    width = 800,
    xaxis=dict(
        title_text = "<b>领域半径(eps)</b>",
        showline=True,
        showgrid=False,
        showticklabels=True,
        linecolor='rgb(204, 204, 204)',
        linewidth=2,
        ticks='outside',
        tickfont=dict(
            family='Arial',
            size=12,
            color='rgb(82, 82, 82)',
        ),
    ),
)
fig.update_yaxes(title_text="<b>聚类数量</b> ")
fig.show()

In [40]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(x = eps, y = noise_num,
                         name='噪声数量',
                         mode='lines',
                         line=dict(color=colors[0],width=line_size[0]),
                         ),
                         )
fig.update_layout(
    width = 800,
    xaxis=dict(
        showline=True,
        showgrid=False,
        showticklabels=True,
        linecolor='rgb(204, 204, 204)',
        linewidth=2,
        ticks='outside',
        tickfont=dict(
            family='Arial',
            size=12,
            color='rgb(82, 82, 82)',
        ),
    ),
)
fig.show()

## NMI, AMI, F-score, Accuracy

In [48]:
dbscan = DBSCAN(eps=76, min_samples=5)
cluster = dbscan.fit(df_cluster)

In [51]:
set(cluster.labels_)
df_result = EliminateNoise(df_cluster,cluster.labels_)
df_result

,Dur,A_count,Count,label
0,21.83,6.0,147.0,0
1,17.02,2.0,37.0,0
2,17.02,2.0,19.0,0
3,17.02,2.0,47.0,0
4,17.02,3.0,11.0,0
...,...,...,...,...
15610,0.00,1.0,2.0,0
15611,0.00,1.0,2.0,0
15612,1.93,1.0,8.0,0
15613,0.03,1.0,3.0,0


In [53]:
set(df_result.label)

{0, 1, 2}